# Student Dropout | Model Development Report

## Part I. Candidates and Assumptions


| Question / goal | Approaches and rationale | Most critical assumption |
|---|---|---|
| Q1. Enrolment differences between graduates and dropouts, across programmes (observed sample). | Group contrasts and course-stratified summaries: directly describe observed students. | Correct measurement and subgroup denominators. |
| Q2 (narrowed). Enrolment associations among eligible entrants to the 17 programmes, 2008/09-2018/19. | Multinomial logistic regression: adjusted associations while retaining Enrolled. | Population coverage and valid uncertainty. |
| Q3 (revised). Predict dropout by normal course-end from information available at enrolment. | Baseline; ridge logistic; GAM; forest; boosting: compare additive, smooth and interaction-based signals.  | Inputs known at prediction; comparable future cases. |
| Q4 (decision). How many more recorded Dropout cases does risk-based review identify than random selection at equal capacity? | Fold-wise top-risk lists at 5%, 10%, 20%; exact random expectation and conditional uncertainty. | Honest out-of-fold ranking; transportability for future use. |

**Data and status.** Predict Students’ Dropout and Academic Success, UCI 697: https://doi.org/10.24432/C5MC89 [Realinho et al., 2022](https://doi.org/10.3390/data7110146). Historical EDA in main.ipynb: 4,424 students, 36 predictors, including Admission grade and Previous qualification (grade); outcomes: 1,421 Dropout / 2,209 Graduate / 794 Enrolled. Historical EDA has been reproduced; fitted results are in Part II. This part records the modelling rationale, updated for Q4.

## I.1 Prediction

- **Revised Q3.** Replace the original semester-updated question with enrolment-time prediction of recorded dropout by normal course-end. Y = 1 for Dropout; Y = 0 for Graduate or Enrolled at that horizon. Enrolled is not eventual graduation. Semester-updated/eventual risk requires dated histories and follow-up; course-specific horizons limit comparability.

- **Shared inputs.** Use enrolment demographics, prior education, application/admission variables and Course. Encode nominal codes categorically; treat parental education 34 and occupation 99 as unknown/blank and check sensitivity. Exclude semester results and unverified financial/macroeconomic snapshots. Learn preprocessing inside training folds.

| Candidate: why it fits / fitting plan | Assumptions, evidence → consequence → repair | Verdict |
|---|---|---|
| B0. Prevalence baseline: training-fold mean(Y), the constant Bernoulli log-loss optimum; no tuning. | No homogeneity claim: intentionally omit X. Low variance but subgroup bias; covariate models may improve it. Full-sample 32.12% is descriptive, not a test score. | Assumptions plausible for a constant benchmark. |
| P1. Ridge logistic [le Cessie & van Houwelingen, 1992](https://doi.org/10.2307/2347628): Bernoulli logit + L2; scale continuous inputs and tune penalty. Stabilizes expanded categorical designs. | Approximately linear, additive log odds; unverified. Missed curvature causes bias, rare levels variance. Check calibration; repair with limited splines/interactions. | Assumptions plausible provisionally. |
| P2. Logistic GAM [Wood, 2008](https://doi.org/10.1111/j.1467-9868.2007.00646.x): penalized smooths for age and both grades + categorical effects; tune smoothing, cap bases at six. | Smooth/additive effects. Resolved-case age-bin dropout shares rise to 70.25%, then fall to 61.42% (cell 16): hypothesis, not conditional proof. Omitted interactions bias; weak smoothing inflates variance. Penalize or add justified interactions. | Assumptions plausible provisionally. |
| P3. Random forest [Breiman, 2001](https://doi.org/10.1023/A:1010933404324): 500 bootstrap trees, Gini splits, averaged probabilities; tune depth, leaf size and feature subsampling. | Thresholds/interactions with adequate local support. Course V = 0.349; smallest resolved group n = 9 (cells 23/27): inclusion evidence, not interactions. Sparse leaves inflate variance; stepwise fits bias smooth effects. Enlarge leaves, restrict depth. | Assumptions plausible with regularization. |
| P4. Shallow boosting [Friedman, 2001](https://doi.org/10.1214/aos/1013203451): sequentially minimize Bernoulli log loss; tune depth 1-3, leaf size, learning rate and iterations. | Stable low-order interactions could reduce additive bias; not demonstrated. Excess updates cause variance; too-shallow trees retain bias. Use shrinkage and inner-fold stopping. | Assumptions plausible with regularization. |




## I.1 Prediction (continued)

- **Validation and uncertainty: all candidates.** Part II: reserve a test set; use identical stratified outer five-fold / inner three-fold splits within training data only; tune by log loss [Cawley & Talbot, 2010](https://jmlr.org/papers/v11/cawley10a.html). Keep feature selection, stopping and calibration inside outer-train. Also report Brier score, calibration, PR-AUC, precision/recall/F1. Q4 fixes hypothetical capacities 5%, 10%, 20% before its analysis; none is a known institutional capacity. Report paired outer losses; folds are not independent CI replicates. Leave test-set comparison to Assure. Future-cohort evaluation can add paired bootstrap intervals, with cluster resampling if justified.

- **Expected ranking.** Ridge > GAM > boosting > forest > baseline for lower log loss, with low confidence: moderate n and unconfirmed interactions favour lower variance. Promote GAM for stable curvature gains; promote trees for reproducible interaction gains and calibration. Treat adjacent ranks as tentative; even baseline may win. Pre-specify a practically meaningful gain before tuning; none is yet supplied.

- **Shared limitations.** Earlier full-sample exploration limits claims of untouched evaluation. Drift biases risk and dependence invalidates uncertainty; future-cohort validation remains unavailable. Original semester-updated Q3 is not currently estimable; missing histories are a data limitation, not an observed assumption violation.

## I.2 Inference

- **Q1 — empirical group contrasts.** Compare age, both grades, Gender and admission/qualification categories within Course for 3,630 graduates/dropouts, using means/medians and proportions. Saved EDA: mean age 26.07 vs 21.78 (cell 14); age-dropout r = 0.267, 95% CI [0.237, 0.297] (cell 26); course Wilson intervals in cell 27. These are unadjusted associations. Report denominators; verify timing/codes to avoid distorted contrasts. Sparse groups limit generalization. Fixed-sample summaries need no sampling CI; existing intervals require population/independence assumptions. Verdict: Assumptions plausible for sample description only.

- **Q2 — multinomial logistic regression [Agresti, 2013](https://mybiostats.wordpress.com/wp-content/uploads/2015/03/3rd-ed-alan_agresti_categorical_data_analysis.pdf).** Target eligible entrants to the 17 programmes during 2008/09-2018/19 under source inclusion rules, not all/future students. Keep all three outcomes; Graduate is the reference. Fit multinomial likelihood with pre-specified Course, Gender, an age spline and the two grade variables, to estimate associations conditional on this stated adjustment set, not causal effects. Report adjusted Dropout/Graduate relative-probability ratios and standardized probability contrasts. No predictive tuning; exploratory after prior EDA, not confirmatory variable selection.

- **Q2 assumptions → repair → verdict.** Correct logit functions, adequate cells and population coverage are required. Misspecification/selection cause bias; sparse cells increase variance; dependence invalidates intervals. Check fit/separation; use 95% intervals under an explicit sampling model and unit/cluster bootstrap for contrasts. Coverage and dependence for the narrowed population remain unverified; inclusion/cohort metadata are unavailable, so restrict claims. Verdict: Assumptions plausible as a working outcome model; population applicability unverified, so population claims are deferred.

## I.3 Causality / decision

- **Revised Q4 (decision).** Among incoming students, how many more students subsequently recorded as dropouts by the end of the normal course duration would be identified by selecting the highest predicted-risk students for additional review, compared with random selection at the same review capacity? Limited review slots make concentration of recorded outcomes useful to assess. The original tutoring/advising effect is unidentified without assignment/exposure data; this explicitly replaces it with a non-causal allocation question. Decision Q4 is the confirmed revised project scope.

- **Method.** Reuse each fixed B0/P1-P4 rule (P1B as sensitivity), with enrolment-only nested-CV OOF scores. Within each validation fold choose k=floor(qN), q=5%,10%,20%; sort by risk, then SHA256(1481:row_id) for ties. Compare H selected Dropout with kD/N for uniform random selection without replacement. Sum H, k and expectations across folds; report precision, recall, lift and extra cases. No Q4 winner selection or new optimizer. Exact random-reference uncertainty is conditional on these cohorts; a fixed-score pair bootstrap gives composition sensitivity, not full training/selection uncertainty. Folds are not independent replicates.

- **Assumptions, consequences, repairs, verdict.** Correct outcome timing and leak-free scores are plausible under the retained pipeline. Leakage or selecting a CV winner inflates apparent gain: use nested predictions and show all fixed rules. Small lists/ties increase instability: deterministic outcome-blind ties, capacity sensitivity and qualified uncertainty. Drift, unknown clustering and different programme horizons limit transportability: restrict conclusions to retrospective CV and require future validation before operational use. Enrolled is not eventual graduation; identified cases are not prevented dropouts, and high risk is not treatment benefit. **Verdict: assumptions plausible for retrospective selection assessment; future policy benefit unverified.** This revised scope does not estimate the original causal effect or satisfy the numerical causal-estimate clause; the distinction is documented, not an unresolved modelling task.
